In [8]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

import joblib
import random
import numpy as np
from scipy.stats import linregress
import torch
from matplotlib import pyplot as plt
from syd import make_viewer, Viewer
from tqdm import tqdm

from vrAnalysis.database import get_database
from vrAnalysis.helpers import (
    Timer, sort_by_preferred_environment, edge2center, beeswarm, format_spines, insert_nans_at_gaps, cross_validate_trials, save_figure, add_scaled_limits, get_placefield_location,
)
from vrAnalysis.sessions import B2Session, SpksTypes
from vrAnalysis.processors import SpkmapProcessor
from vrAnalysis.processors.support import median_zscore
from vrAnalysis.processors.placefields import get_placefield, get_frame_behavior, get_placefield_prediction
from dimilibi import measure_r2, mse
from dimilibi.pca import PCA
from dimilibi import gaussian_filter, fit_powerlaw_decay, fit_powerlaw_derivatives
from dimensionality_manuscript.registry import PopulationRegistry, get_model, ModelName, short_model_name
from dimensionality_manuscript.workflows.measure_cvpca import get_filepath as get_cvpca_filepath
from dimensionality_manuscript.workflows.measure_cvpca import nanmax
from dimensionality_manuscript import StimSpaceConfig, get_data_config, ResultsStore, ResultsAggregator

plt.rcParams["font.size"] = 18

# get session database
sessiondb = get_database("vrSessions")

# get population registry and models
registry = PopulationRegistry()

In [9]:
cfg = StimSpaceConfig()
sessions = sessiondb.iter_sessions(imaging=True)
store = ResultsStore()
results = ResultsAggregator(cfg, store, sessions)

In [16]:
class StimSpaceViewer(Viewer):
    def __init__(self):
        self = cfg.build_syd(self, results)
        self.update_selection("view_by", value="mouse_average")
        self.add_boolean("norm_to_1", value=False)
        self.add_integer("start_idx", min=1, max=100, value=10)
        self.add_integer("end_idx", min=1, max=100, value=20)
        self.add_selection("xscale", options=["linear", "log"], value="log")
        add_scaled_limits(self, max_value=1e8, min_log_exponent=-10, log_default=True)

    def plot(self, state):
        result, axes_names = self.get_result(state)

        _keys = list(result.keys())
        if result[_keys[0]].ndim == 1:
            result = {k: v[None] for k, v in result.items()}

        spectrum = np.sqrt(np.maximum(result["cv_variance_squared_placefield_placefield"], 0.0))

        kws = dict(start_idx=state["start_idx"], end_idx=state["end_idx"])
        alphas = np.array([fit_powerlaw_decay(spectrum[i], **kws, ignore_nans=True)[0] for i in range(len(spectrum))])

        norm = lambda x: x / np.nansum(x, axis=-1, keepdims=True) if state["norm_to_1"] else x

        beewidth = 0.25
        xvals = np.arange(spectrum.shape[-1]) + 1
        fig, ax = plt.subplots(1, 2, figsize=(7, 4), width_ratios=[1, 0.5], layout="constrained")
        ax[0].plot(xvals, norm(spectrum).T)
        ax[0].set_xlabel("Dimension")
        ax[0].set_ylabel("Covariance")
        ax[0].set_xscale(state["xscale"])
        ax[0].set_yscale(state["yscale"])
        if len(alphas) > 1:
            ax[1].plot(beewidth * beeswarm(alphas), alphas, "o", color="k")
        else:
            ax[1].plot(0, alphas, "o", color="k")
        format_spines(ax[1], x_pos=-0.05, y_pos=-0.05, spines_visible=["left"], xticks=[])
        ax[1].set_xlim([-0.5, 0.5])
        ax[1].set_ylabel(f"Alpha {state['start_idx']}-{state['end_idx']}")
        return fig


viewer = StimSpaceViewer()
viewer.show()